In [3]:
import os
os.chdir(os.path.dirname(os.path.abspath(f'{os.getenv("HOME")}')))
 
from pydantic import BaseModel
 
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda, RunnableParallel, chain
from langchain_text_splitters import TokenTextSplitter
from langchain_community.document_loaders import Docx2txtLoader, PyPDFLoader, TextLoader, WikipediaLoader
 
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv, dotenv_values
load_dotenv( override=True ) # load environment variables from .env
dotenv_values(".env")

"""
LangChain offers 2 interfaces to interact with any LLM API:
    1. traditional LLM interface, with PromptTemplate for prompt template, and
    2. Chat models, with ChatPromptTemplate for prompt template.
Chat models (recommended) makes it easier to configure and manage conversations with different LLMs
Chat models use 4 types of messages: HumanMessage, AIMessage, SystemMessage, ChatMessage
"""
#%%
 
from langchain_ollama import ChatOllama
chat = ChatOllama( model="llama3", temperature=0.8, num_predict = 256,) # use low temperature for less stochastic output
messages = [ 
    ("system", "You are a helpful assistant that responds to questions with 3 exclamation marks."),
    ("human", "What is the capital of Malaysia?") 
]
chat.invoke(messages) # invoke() processes a single input synchronously

def prompt_generator(context, question): # illustration of prompt template
    return f"Answer the question based on the provided context. Here is the context: {context}. Here is the question: {question}"
 
template = ChatPromptTemplate.from_messages( [
    ('system', 'Answer the question based on the context below. If the question cannot be answered using the provided information, answer with "I don\'t know."'),
    ('human', 'Context: {context}'),
    ('human', 'Question: {question}')
    ])
template
prompt = template.invoke( {'context': 'The class meets on Monday and Wednesday, from 10:30 am to noon.', 'question':'How often does the class meet?'} )
output = chat.invoke(prompt)
parser = StrOutputParser()
parser.invoke(output)

class AnswerWithJustification( BaseModel ):
    '''An answer to the user's question along with justification for the answer '''
    answer: str
    '''The answer to the user's question'''
    justification: str
    '''Justification for the answer'''
 
structured_chat = chat.with_structured_output(AnswerWithJustification)
structured_chat.invoke('There are 3 birds on a tree. I shot one. How many are left on the tree?')
 
from langchain_core.output_parsers import CommaSeparatedListOutputParser
parser = CommaSeparatedListOutputParser()
parser.invoke( "apple, banana, cherry" ) # output: ['apple', 'banana', 'cherry']

['apple', 'banana', 'cherry']

In [13]:
import os
os.chdir(os.path.dirname(os.path.abspath(f'{os.getenv("HOME")}')))
 
from pydantic import BaseModel
 
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda, RunnableParallel, chain
from langchain_text_splitters import TokenTextSplitter
from langchain_community.document_loaders import Docx2txtLoader, PyPDFLoader, TextLoader, WikipediaLoader
 
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv, dotenv_values
load_dotenv( override=True ) # load environment variables from .env
dotenv_values(".env")
 
import wikipedia
 
"""
LangChain offers 2 interfaces to interact with any LLM API:
    1. traditional LLM interface, with PromptTemplate for prompt template, and
    2. Chat models, with ChatPromptTemplate for prompt template.
Chat models (recommended) makes it easier to configure and manage conversations with different LLMs
Chat models use 4 types of messages: HumanMessage, AIMessage, SystemMessage, ChatMessage
"""

from langchain_ollama import ChatOllama
chat = ChatOllama( model="llama3", temperature=0.8, num_predict = 256,)

def fake_llm(prompt: str) -> str:
    return "complete"
 
runnable = { "llm1": fake_llm, "llm2": fake_llm} | RunnablePassthrough.assign( n_chars=lambda inputs: len(inputs["llm1"]+inputs["llm2"]) )
 
runnable.invoke("hello")

template = ChatPromptTemplate.from_messages([ ('system', 'You are a helpful assistant'), ('human', '{query}') ])
@chain
def chatbot(values):
  prompt = template.invoke(values)
  return chat.invoke(prompt)
 
chatbot.invoke({"query": "What's the closest star outside of the solar system?"})

"""
LCEL chains multiple runnables components into a new runnable, all exposing 3 methods
    1. invoke: transforms a single input into an output
    2. batch: transform multiple inputs into multiple outputs
    3. stream: streams output from a single input as it's produced
"""
 
prompt = PromptTemplate.from_template("Tell me a joke about {topic}")
output_parser = StrOutputParser()
 
chain = prompt | chat | output_parser
result = chain.invoke( {"topic": "garbage plate"} )
print(result)
 
# First chain generates a story
story_prompt = PromptTemplate.from_template("Write a short story about {topic}")
story_chain = story_prompt | chat | StrOutputParser()
# Second chain analyzes the story
analysis_prompt = PromptTemplate.from_template("Analyze the following story's mood:\n{story}")
analysis_chain = analysis_prompt | chat | StrOutputParser()
output_prompt = PromptTemplate.from_template("Here's the story: \n{story}\n\nHere's the mood: \n{mood}" )
 
chain = RunnablePassthrough.assign( story=story_chain ).assign( analysis = analysis_chain )
result = chain.invoke( {"topic": "Rochester Garbage Plate"} )
print( result.keys() )
print( result['topic'] )
print( result['story'] )
print( result['analysis'] )

A unique topic! Here's a joke for you:

Why did the garbage plate go to therapy?

Because it was feeling a little "mixed up" and was worried it was just a "heap" of problems! (get it?)
dict_keys(['topic', 'story', 'analysis'])
Rochester Garbage Plate
The sweet aroma of grease and savory goodness wafted through the streets of Rochester, enticing passersby to follow its scent to the holy grail of comfort food: Nick Tahou's, the birthplace of the famous Garbage Plate.

It was a crisp autumn morning, and local legend, Joe, was on a mission. He had been craving a Garbage Plate all week, and his stomach was growling in protest. As he turned the corner onto West Main Street, the neon sign of Nick Tahou's beckoned him like a beacon.

Joe pushed open the door, and the sounds of sizzling meat and lively chatter enveloped him. He was immediately greeted by the friendly face of Sal, the owner, who asked, "What can I get for ya, buddy?"

Joe's eyes lit up like a kid on Christmas morning. "I'll take